# PDFs with*out* OCR text

In [ ]:
!pip install pdf2image

In [ ]:
from pdf2image import convert_from_path

images = convert_from_path('./bilateral.pdf')

for i, image in enumerate(images):
    image.save('./images/bilateral_pg' + str(i) + '.jpg', 'JPEG')

Need to install `tesseract` on system before we can start:

https://tesseract-ocr.github.io/tessdoc/Installation.html


In [ ]:
!pip install pytesseract

In [ ]:
import pytesseract

from PIL import Image
import pytesseract

img = Image.open("./images/bilateral_pg0.jpg") # Open the image with pillow
img.load()
text = pytesseract.image_to_string(img, lang='ind') # Extract image's text
print(text)

with open("output.txt", "w") as text_file:
    text_file.write(text)

# Manipulating hierarchical data in Pandas

## Creating table from singluar CSV/XLSX

In [ ]:
import pandas as pd

# started with skiprows=2, but that
imports_df = pd.read_excel('BPS_import2026.xlsx', skiprows=4)
imports_df

In [ ]:
# referencing our spreadsheet, we know that the last column is totals for each category so we can take 2 steps here:
# 1) reduce the columns to code/year/total
# 2) rename the last column to something 

imports_df = imports_df[['HS Code', 'Year', 'Unnamed: 30']].copy()
imports_df = imports_df.rename(columns={"Unnamed: 30": "Totals"})
imports_df

## Creating table from multiple like CSV/XLSXs

In [12]:
import os
import pandas as pd

# ./ is saying that the folder, imports, is in the same directory as this notebook
folder_path = './imports'

# empty dataframe to add to/combine tables in
merged_imports_df = pd.DataFrame(columns=['HS Code','Year','Totals'])

for file in os.listdir(folder_path):
    # we do this check bc some OS include "hidden" files that could trip your code up like .DS_Store on MacOS
    if file.endswith('.xlsx'):

        # Step 1: get spreadsheet into consistent format
        file_path = "./imports/" + file
        imports_df = pd.read_excel(file_path, skiprows=4)

        # get name of last column since it varies between tables
        last_column = imports_df.columns[-1]
        imports_df = imports_df[['HS Code', 'Year', last_column]].copy()
        imports_df = imports_df.rename(columns={last_column: "Totals"})
        
        # Step 2: merge spreadsheet into combined dataframe
        merged_imports_df = pd.concat([merged_imports_df, imports_df], ignore_index=True)


merged_imports_df.to_csv("merged_imports_df.csv")

In [17]:
test = merged_imports_df.pivot_table(index = ['HS Code'], columns='Year', values = 'Totals', aggfunc='sum').reset_index()
test.to_csv("merged_imports_df.csv")